# Phase 2 — Day 1, Notebook 03  
# Document Ingestion, Chunking & Metadata — Made Visible

**Hands-on outcome:** Build the policy ingestion layer and *visibly compare* multiple chunking strategies.

###
- Four chunking methods rather than only two.
- DataFrames that show document → page → chunk → metadata relationships.
- Chunk-size statistics and side-by-side previews.
- A practical explanation of when each strategy helps or hurts retrieval.

> ** “Before we discuss embeddings, we need to see exactly what the retriever will be allowed to search.”

## 1. Load the healthcare policy corpus and make the source structure visible

In [ ]:
import sys
print(sys.executable)

import numpy as np
import pandas as pd
import pyarrow as pa

print("NumPy   :", np.__version__)
print("Pandas  :", pd.__version__)
print("PyArrow :", pa.__version__)

In [ ]:
from pathlib import Path
import json, re
import pandas as pd
import numpy as np
from pypdf import PdfReader
from IPython.display import display

ROOT = Path(".")
POLICY_DIR = ROOT / "data" / "healthcare_policies"
ARTIFACT_DIR = ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

manifest = json.loads((POLICY_DIR / "policy_manifest.json").read_text(encoding="utf-8"))
manifest_df = pd.DataFrame(manifest)

display(
    manifest_df[
        ["doc_id","title","plan_type","policy_domain","effective_date","filename"]
    ].rename(columns={
        "doc_id":"Document ID",
        "title":"Policy",
        "plan_type":"Plan",
        "policy_domain":"Domain",
        "effective_date":"Effective Date",
        "filename":"Source File"
    })
)

**How to read the output:** One business topic can appear in more than one policy.  
The metadata columns are what later let us restrict retrieval to the correct plan, domain and policy version.

> ** Similar text is not necessarily the same business rule.

## 2. Extract PDF pages and show the document → page relationship

In [ ]:
def extract_pdf_pages(pdf_path):
    reader = PdfReader(str(pdf_path))
    return [
        {"page": page_no, "text": page.extract_text() or ""}
        for page_no, page in enumerate(reader.pages, start=1)
    ]

docs = {}
page_rows = []

for item in manifest:
    pages = extract_pdf_pages(POLICY_DIR / item["filename"])
    docs[item["doc_id"]] = {"metadata":item, "pages":pages}
    for p in pages:
        page_rows.append({
            "doc_id":item["doc_id"],
            "plan_type":item["plan_type"],
            "page":p["page"],
            "characters":len(p["text"]),
            "words":len(p["text"].split()),
            "preview":re.sub(r"\s+"," ",p["text"])[:120] + "..."
        })

page_df = pd.DataFrame(page_rows)
display(page_df)

The table makes the ingestion unit explicit: a PDF contains pages, and each page contains text that will subsequently be chunked.

> **Takeaway:** Source/page lineage should be attached *before* chunking so it survives every downstream transformation.

## 3. Minimal preprocessing — preserve business structure

In [ ]:
def clean_text(text):
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

sample_doc_id = "GOLD-PPO-2026"
sample_text = "\n".join(clean_text(p["text"]) for p in docs[sample_doc_id]["pages"])

comparison_df = pd.DataFrame([
    {
        "Version":"Raw extracted text",
        "Characters":sum(len(p["text"]) for p in docs[sample_doc_id]["pages"]),
        "Preview":re.sub(r"\s+"," ",docs[sample_doc_id]["pages"][0]["text"])[:180] + "..."
    },
    {
        "Version":"Minimally cleaned text",
        "Characters":len(sample_text),
        "Preview":re.sub(r"\s+"," ",sample_text)[:180] + "..."
    }
])
display(comparison_df)

> **** Cleaning should remove extraction noise, not policy meaning. Over-cleaning can delete headings, qualifiers or table structure that retrieval later needs.

## 4. Four chunking strategies

We will compare:

1. **Fixed-size + overlap** — predictable size, simple implementation.
2. **Sentence-based** — tries not to split sentences.
3. **Recursive separator-based** — prefers paragraph/sentence boundaries but falls back to smaller separators.
4. **Section-aware** — preserves policy headings and business meaning.

In [ ]:
def fixed_chunks(text, chunk_chars=650, overlap=100):
    chunks, start = [], 0
    while start < len(text):
        end = min(len(text), start + chunk_chars)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks

def sentence_chunks(text, target_chars=650):
    sentences = re.split(r"(?<=[.!?])\s+", re.sub(r"\s+"," ",text).strip())
    chunks, current = [], ""
    for sentence in sentences:
        candidate = (current + " " + sentence).strip()
        if current and len(candidate) > target_chars:
            chunks.append(current)
            current = sentence
        else:
            current = candidate
    if current:
        chunks.append(current)
    return chunks

def recursive_chunks(text, target_chars=650, overlap=80):
    separators = ["\n\n", "\n", ". ", "; ", ", ", " "]

    def split_recursive(segment, level=0):
        if len(segment) <= target_chars or level >= len(separators):
            return [segment.strip()] if segment.strip() else []

        sep = separators[level]
        parts = segment.split(sep)
        if len(parts) == 1:
            return split_recursive(segment, level + 1)

        result, current = [], ""
        joiner = sep if sep.strip() else " "
        for part in parts:
            candidate = (current + joiner + part).strip() if current else part.strip()
            if current and len(candidate) > target_chars:
                result.extend(split_recursive(current, level + 1))
                tail = current[-overlap:] if overlap else ""
                current = (tail + joiner + part).strip()
            else:
                current = candidate
        if current:
            result.extend(split_recursive(current, level + 1))
        return result

    return split_recursive(text)

SECTION_PATTERN = re.compile(r"(SECTION\s+\d+:\s+[A-Z &\-]+)", re.IGNORECASE)

def section_chunks(text):
    parts = SECTION_PATTERN.split(text)
    chunks, current_heading = [], "DOCUMENT HEADER"

    for part in parts:
        part = part.strip()
        if not part:
            continue
        if SECTION_PATTERN.fullmatch(part):
            current_heading = part.upper()
        else:
            chunks.append({"section":current_heading, "text":part})
    return chunks

strategy_samples = {
    "Fixed + overlap": fixed_chunks(sample_text),
    "Sentence-based": sentence_chunks(sample_text),
    "Recursive": recursive_chunks(sample_text),
    "Section-aware": [x["text"] for x in section_chunks(sample_text)]
}

strategy_summary = []
for strategy, chunks in strategy_samples.items():
    lengths = [len(c) for c in chunks]
    strategy_summary.append({
        "Chunking Strategy":strategy,
        "No. of Chunks":len(chunks),
        "Avg Chars":round(np.mean(lengths),1),
        "Median Chars":round(np.median(lengths),1),
        "Min Chars":min(lengths),
        "Max Chars":max(lengths),
        "Avg Words":round(np.mean([len(c.split()) for c in chunks]),1)
    })

display(pd.DataFrame(strategy_summary))

### What the comparison means

- **Fewer, larger chunks** preserve surrounding context but can retrieve unnecessary information.
- **More, smaller chunks** can improve precision but may separate a rule from its condition.
- **Overlap** reduces boundary loss but duplicates text and increases index size.
- **Section-aware chunks** preserve business meaning when the document has reliable headings.

> **** There is no universally “best” chunk size. The right strategy is the one that produces the best retrieval and answer metrics on your evaluation set.

## 5. Make the actual chunk boundaries visible

In [ ]:
preview_rows = []
for strategy, chunks in strategy_samples.items():
    for i, chunk in enumerate(chunks[:3], start=1):
        preview_rows.append({
            "Strategy":strategy,
            "Chunk #":i,
            "Characters":len(chunk),
            "Words":len(chunk.split()),
            "Starts With":re.sub(r"\s+"," ",chunk)[:220] + ("..." if len(chunk)>220 else "")
        })

display(pd.DataFrame(preview_rows))

Use this output to point at where the *same policy* is cut differently by each method.

> **Teaching question:** “If the authorization threshold is at the end of one chunk and the exception is at the start of the next, what could go wrong?”

## 6. Build all four chunk corpora with source metadata

In [ ]:
def build_chunk_corpus(strategy):
    records = []

    for doc_id, doc in docs.items():
        meta = doc["metadata"]

        for page_info in doc["pages"]:
            text = clean_text(page_info["text"])

            if strategy == "fixed":
                chunks = [{"section":"FIXED_WINDOW","text":t} for t in fixed_chunks(text)]
            elif strategy == "sentence":
                chunks = [{"section":"SENTENCE_GROUP","text":t} for t in sentence_chunks(text)]
            elif strategy == "recursive":
                chunks = [{"section":"RECURSIVE_WINDOW","text":t} for t in recursive_chunks(text)]
            elif strategy == "section":
                chunks = section_chunks(text)
            else:
                raise ValueError("Unsupported strategy")

            for idx, chunk in enumerate(chunks):
                records.append({
                    "chunk_id":f"{doc_id}-{page_info['page']:02d}-{strategy}-{idx:03d}",
                    "doc_id":doc_id,
                    "source_file":meta["filename"],
                    "title":meta["title"],
                    "plan_type":meta["plan_type"],
                    "policy_domain":meta["policy_domain"],
                    "effective_date":meta["effective_date"],
                    "page":page_info["page"],
                    "section":chunk["section"],
                    "chunk_strategy":strategy,
                    "char_count":len(chunk["text"]),
                    "word_count":len(chunk["text"].split()),
                    "text":chunk["text"]
                })

    return records

corpora = {
    "fixed":build_chunk_corpus("fixed"),
    "sentence":build_chunk_corpus("sentence"),
    "recursive":build_chunk_corpus("recursive"),
    "section":build_chunk_corpus("section")
}

corpus_comparison = []
for strategy, records in corpora.items():
    corpus_comparison.append({
        "Strategy":strategy,
        "Total Chunks":len(records),
        "Avg Chars":round(np.mean([r["char_count"] for r in records]),1),
        "Min Chars":min(r["char_count"] for r in records),
        "Max Chars":max(r["char_count"] for r in records),
        "Documents Covered":len(set(r["doc_id"] for r in records)),
        "Plans Represented":len(set(r["plan_type"] for r in records))
    })

display(pd.DataFrame(corpus_comparison))

## 7. Show the chunk → metadata → source relationship

In [ ]:
relationship_df = pd.DataFrame(corpora["section"])[
    ["chunk_id","doc_id","plan_type","section","page","char_count","source_file","text"]
].copy()

relationship_df["text"] = relationship_df["text"].str.replace(r"\s+"," ",regex=True).str[:180] + "..."
display(relationship_df.head(12))

This is the key table for explaining enterprise retrieval:

**Chunk text** is what similarity search operates on.  
**Metadata** is what constrains and governs retrieval.  
**Source fields** are what enable traceability and citations.

## 8. Compare how a single business rule appears across strategies

In [ ]:
needle = "physical therapy"

rows = []
for strategy, records in corpora.items():
    matches = [r for r in records if needle in r["text"].lower()]
    for r in matches[:2]:
        rows.append({
            "Strategy":strategy,
            "Chunk ID":r["chunk_id"],
            "Plan":r["plan_type"],
            "Section":r["section"],
            "Chars":r["char_count"],
            "Policy Text":re.sub(r"\s+"," ",r["text"])[:260] + "..."
        })

display(pd.DataFrame(rows))

> **** The underlying policy has not changed — only the unit presented to the retriever has changed. That design choice can change which evidence is found later.

## 9. Persist all chunk corpora for the embedding notebook

In [ ]:
def save_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in records:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

for strategy, records in corpora.items():
    path = ARTIFACT_DIR / f"policy_chunks_{strategy}.jsonl"
    save_jsonl(records, path)

saved = pd.DataFrame([
    {"Strategy":s, "Artifact":str(ARTIFACT_DIR / f"policy_chunks_{s}.jsonl"), "Chunks":len(r)}
    for s,r in corpora.items()
])
display(saved)

## Day 1 checkpoint

You can now visibly explain:

**PDF → Page → Chunk → Metadata → Source lineage**

and compare:

**Fixed vs Sentence vs Recursive vs Section-aware chunking**

The next notebook asks: *Which chunks are actually closest to a user's question, how is similarity interpreted, and how does ranking change after reranking?*